<a href="https://colab.research.google.com/github/ghduf0201-oss/GPT2.0-0toHero/blob/main/notebook_03_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — MLP on Tiny Shakespeare

이제 모델은 2번 MLP를 그대로 두고, **데이터만 `tiny Shakespeare`로 바꿉니다.**

문제는 여전히 **fixed context -> next char** 입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print("text length:", len(text))
print("vocab_size:", vocab_size)

text length: 1115394
vocab_size: 65


## 1. Sliding-window dataset

In [ ]:
class CharSequenceNextCharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + self.block_size]
        return x, y

block_size = 16
dataset = CharSequenceNextCharDataset(data, block_size)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)
print("decoded x:", ''.join(itos[i.item()] for i in xb[0]))
print("decoded y:", itos[yb[0].item()])

xb.shape: torch.Size([128, 16])
yb.shape: torch.Size([128])
decoded x: n of a gentlewom
decoded y: a


## 2. MLP model

In [ ]:
class MLPCharacterModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Embedding(vocab_size, emb_dim),
            nn.Flatten(),
            nn.Linear(block_size * emb_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, x):
        return self.net(x)

model = MLPCharacterModel(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([128, 65])
initial loss: 4.231189250946045


## 3. 학습

In [ ]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLPCharacterModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(10):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.6024
epoch  1 | train loss 2.2616
epoch  2 | train loss 2.1240
epoch  3 | train loss 2.0605
epoch  4 | train loss 1.9947
epoch  5 | train loss 1.9559
epoch  6 | train loss 1.9346
epoch  7 | train loss 1.8903
epoch  8 | train loss 1.8760
epoch  9 | train loss 1.8604


## 4. Sampling

In [ ]:
@torch.no_grad()
def sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = [0] * block_size
    for ch in start_text:
        if ch in stoi:
            context = context[1:] + [stoi[ch]]
    out = list(start_text)
    for _ in range(max_new_tokens):
        x = torch.tensor([context], dtype=torch.long, device=device)
        logits = model(x)
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1).item()
        out.append(itos[ix])
        context = context[1:] + [ix]
    return "".join(out)

print(sample_mlp(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

ROMEO: Benexchmith Comarealour.
Thy gorthed betcemprid, incs as mystlfof?

NEMARETHOLO:
Ay, sees blood
Wis pror acmeds
The is grom him, my soar it that stanthyr,
Alaze abouty oken thule enchent held my lat

Therd-my gut not conts bo hos the would
Thou sold privones bry.

LINGbe$: Bfou, oup is tell condtty; whepe o'd on tomithy goojle;
The impelfod bord his kto here'er's no aks sothere?

might caunt on a


## 5. 정리

- 같은 MLP 모델을 더 큰 텍스트에도 적용할 수 있습니다.
- 바뀌는 것은 dataset입니다.
- 하지만 fixed context MLP는 긴 문맥을 잘 다루지 못합니다.